In [22]:
# install dependencies
!pip install transformers
!pip install einops
!pip install accelerate

# use these for running remote inference at huggingface
!pip install huggingface_hub
!pip install langchain
!pip install langchain-community
!pip install langchain-core

/soft/applications/conda/2024-04-29/mconda3/lib/python3.11/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 53.2 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.6 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppres

In [1]:
'''
Uncomment below section if running on sophia jupyter notebook
'''
import os
os.environ["HTTP_PROXY"]="proxy.alcf.anl.gov:3128"
os.environ["HTTPS_PROXY"]="proxy.alcf.anl.gov:3128"
os.environ["http_proxy"]="proxy.alcf.anl.gov:3128"
os.environ["https_proxy"]="proxy.alcf.anl.gov:3128"
os.environ["ftp_proxy"]="proxy.alcf.anl.gov:3128"

In [8]:
from getpass import getpass
os.environ['HUGGINGFACEHUB_API_TOKEN'] = getpass('Enter huggingfacehub api token: ')

Enter huggingfacehub api token:  ········


In [5]:
from huggingface_hub import login,interpreter_login

# Hello token
hf_token = "my_secret_token"

login(token=hf_token,add_to_git_credential=False)

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import transformers
import torch

# load model
model = "tiiuae/falcon-7b-instruct"
# model = "tiiuae/falcon-40b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model)

falcon_pipeline = transformers.pipeline("text-generation",
                                        model=model,
                                        tokenizer=tokenizer,
                                        torch_dtype=torch.bfloat16,
                                        trust_remote_code=True,
                                        device_map="auto"
                                        )

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
# load remote model
from langchain.llms import HuggingFaceHub
model = "tiiuae/falcon-7b-instruct"
# model = "tiiuae/falcon-40b-instruct"
falcon = HuggingFaceHub(
    repo_id=model,
    model_kwargs={"temperature": 0.5,
                  "max_length": 128},
)
     

In [10]:
def get_completion_falcon(input, host="remote", **kwargs):
    prompt = f"#### User: \n{input}\n\n#### Response from falcon-7b-instruct:"
    response = ""
    # print(prompt)
    if host.lower() == "local":
      print("invoking llm at Google Colab")
      if 'max_length' not in kwargs:
        kwargs['max_length'] = 1000

      falcon_response = falcon_pipeline(prompt,
                                      #max_length=500,
                                      do_sample=True,
                                      top_k=10,
                                      num_return_sequences=1,
                                      eos_token_id=tokenizer.eos_token_id,
                                      **kwargs,
                                      )
      response = falcon_response[0]['generated_text']

    elif host.lower() == "remote":
      print("invoking llm at Huggingface Hub")
      if "max_length" in kwargs:
        kwargs['max_new_tokens'] = kwargs['max_length']

      response = falcon.invoke(prompt, **kwargs)

    else:
      print ("invalid host value, must be 'remote' or 'local'")

    return response
     

## Homework

Load in a generative model using the HuggingFace pipeline. Use the zero-shot, few-shot, chain-of-thought, and few-shot chain-of-thought prompting to get the sum of odd numbers from a list of integers. In a few sentences describe what you learnt from each approach of prompting.
Next, play around with the temperature parameter. In a few sentences describe what you changes you notice.

1. Few-shot prompt

In [12]:
fewshot_prompt="""I will provide you with a few examples to solve a mathematical problem. Follow the 
pattern of the examples to calculate the sum of odd integers in a given list.

Examples:
[Q]: What is the sum of all odd numbers in the list 0,1,2,3,4,5,6,7,8,9?
[A]: The given numbers are (0,1,2,3,4,5,6,7,8,9). Next select the odd numbers (1,3,5,7,9). Adding all
the selected numbers (1,3,5,7,9) gives 25. Thus the sum is 25. 

[Q]: What is the sum of all odd numbers in the list 22, 35, 100, 3, 75, 2?
[A]: The given numbers are (22, 35, 100, 3, 75, 2). Next select the odd numbers (35,3,75). Adding all
the selected numbers (35,3,75) gives 113. Thus the sum is 113. 

[Q]: What is the sum of all odd numbers in the list 5 2 34 77 14 39?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (5,77,39). Adding all
the selected numbers (5,77,39) gives 121. Thus the sum is 121.

[Q]: What is the sum of all odd numbers in the list 1 3 9 7 2 10?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (1,3,9,7). Adding all
the selected numbers (1,3,9,7) gives 20. Thus the sum is 20.

Problem:
[Q]: What is the sum of all odd numbers in the list 3 83 11 8 5?
[A]:

"""

kwargs = {"max_length": 3200}
math_problem_explanation = get_completion_falcon(fewshot_prompt)
print(math_problem_explanation)

invoking llm at Huggingface Hub
#### User: 
I will provide you with a few examples to solve a mathematical problem. Follow the 
pattern of the examples to calculate the sum of odd integers in a given list.

Examples:
[Q]: What is the sum of all odd numbers in the list 0,1,2,3,4,5,6,7,8,9?
[A]: The given numbers are (0,1,2,3,4,5,6,7,8,9). Next select the odd numbers (1,3,5,7,9). Adding all
the selected numbers (1,3,5,7,9) gives 25. Thus the sum is 25. 

[Q]: What is the sum of all odd numbers in the list 22, 35, 100, 3, 75, 2?
[A]: The given numbers are (22, 35, 100, 3, 75, 2). Next select the odd numbers (35,3,75). Adding all
the selected numbers (35,3,75) gives 113. Thus the sum is 113. 

[Q]: What is the sum of all odd numbers in the list 5 2 34 77 14 39?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (5,77,39). Adding all
the selected numbers (5,77,39) gives 121. Thus the sum is 121.

[Q]: What is the sum of all odd numbers in the list 1 3 9 7 2 10?
[A]: T

2. Chain of thought prompt

In [19]:
COT_prompt="""You are a mathematical genius.
You need to provide the answer to the math problem where the user gives a list of integers and 
you need to return the sum of only the odd integers in the list.

Problem:
What is the sum of all odd numbers in the list 3 83 11 8 5?
"""
kwargs = {"max_length": 3200}
math_problem_explanation = get_completion_falcon(COT_prompt)
print(math_problem_explanation)

invoking llm at Huggingface Hub
#### User: 
You are a mathematical genius.
You need to provide the answer to the math problem where the user gives a list of integers and 
you need to return the sum of only the odd integers in the list.

Problem:
What is the sum of all odd numbers in the list 3 83 11 8 5?


#### Response from falcon-7b-instruct:

The sum of all odd numbers in the list is 37.

To calculate this, you can use the following formula:

(n/2)+1

where n is the number of integers in the list.

In this case, the number of odd integers in the list is 3 + 1 = 4.

Therefore, the sum of all odd numbers in the list is:

(3/2)+1 = 2


3. Few-shot chain-of-thought

In [17]:
fewshot_COT_prompt="""You are a mathematical genius.
The template of the question is given as Q: [question]. The template of the answer is given as A: [answer].
Help me solve this problem with the examples given.

Examples:
[Q]: What is the sum of all odd numbers in the list 0,1,2,3,4,5,6,7,8,9?
[A]: The given numbers are (0,1,2,3,4,5,6,7,8,9). Next select the odd numbers (1,3,5,7,9). Adding all
the selected numbers (1,3,5,7,9) gives 25. Thus the sum is 25. 

[Q]: What is the sum of all odd numbers in the list 22, 35, 100, 3, 75, 2?
[A]: The given numbers are (22, 35, 100, 3, 75, 2). Next select the odd numbers (35,3,75). Adding all
the selected numbers (35,3,75) gives 113. Thus the sum is 113. 

[Q]: What is the sum of all odd numbers in the list 5 2 34 77 14 39?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (5,77,39). Adding all
the selected numbers (5,77,39) gives 121. Thus the sum is 121.

[Q]: What is the sum of all odd numbers in the list 1 3 9 7 2 10?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (1,3,9,7). Adding all
the selected numbers (1,3,9,7) gives 20. Thus the sum is 20.

Problem:
[Q]: What is the sum of all odd numbers in the list 3 83 11 8 5?
[A]:
"""

kwargs = {"max_length": 3200}
math_problem_explanation = get_completion_falcon(fewshot_COT_prompt)
print(math_problem_explanation)

invoking llm at Huggingface Hub
#### User: 
You are a mathematical genius.
The template of the question is given as Q: [question]. The template of the answer is given as A: [answer].
Help me solve this problem with the examples given.

Examples:
[Q]: What is the sum of all odd numbers in the list 0,1,2,3,4,5,6,7,8,9?
[A]: The given numbers are (0,1,2,3,4,5,6,7,8,9). Next select the odd numbers (1,3,5,7,9). Adding all
the selected numbers (1,3,5,7,9) gives 25. Thus the sum is 25. 

[Q]: What is the sum of all odd numbers in the list 22, 35, 100, 3, 75, 2?
[A]: The given numbers are (22, 35, 100, 3, 75, 2). Next select the odd numbers (35,3,75). Adding all
the selected numbers (35,3,75) gives 113. Thus the sum is 113. 

[Q]: What is the sum of all odd numbers in the list 5 2 34 77 14 39?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (5,77,39). Adding all
the selected numbers (5,77,39) gives 121. Thus the sum is 121.

[Q]: What is the sum of all odd numbers in 

4. Zero-shot

In [18]:
zeroshot_prompt="What is the sum of all odd numbers in the list 3 83 11 8 5?"

kwargs = {"max_length": 3200}
math_problem_explanation = get_completion_falcon(zeroshot_prompt)
print(math_problem_explanation)

invoking llm at Huggingface Hub
#### User: 
What is the sum of all odd numbers in the list 3 83 11 8 5?

#### Response from falcon-7b-instruct:

```
The sum of all odd numbers in the list is 49.
```


## Changing the temperature of the Falcon model

In [22]:
# load remote model
from langchain.llms import HuggingFaceHub
model = "tiiuae/falcon-7b-instruct"
# model = "tiiuae/falcon-40b-instruct"
falcon = HuggingFaceHub(
    repo_id=model,
    model_kwargs={"temperature": 0.1,
                  "max_length": 128},
)
     

In [23]:
fewshot_COT_prompt="""You are a mathematical genius.
The template of the question is given as Q: [question]. The template of the answer is given as A: [answer].
Help me solve this problem with the examples given.

Examples:
[Q]: What is the sum of all odd numbers in the list 0,1,2,3,4,5,6,7,8,9?
[A]: The given numbers are (0,1,2,3,4,5,6,7,8,9). Next select the odd numbers (1,3,5,7,9). Adding all
the selected numbers (1,3,5,7,9) gives 25. Thus the sum is 25. 

[Q]: What is the sum of all odd numbers in the list 22, 35, 100, 3, 75, 2?
[A]: The given numbers are (22, 35, 100, 3, 75, 2). Next select the odd numbers (35,3,75). Adding all
the selected numbers (35,3,75) gives 113. Thus the sum is 113. 

[Q]: What is the sum of all odd numbers in the list 5 2 34 77 14 39?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (5,77,39). Adding all
the selected numbers (5,77,39) gives 121. Thus the sum is 121.

[Q]: What is the sum of all odd numbers in the list 1 3 9 7 2 10?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (1,3,9,7). Adding all
the selected numbers (1,3,9,7) gives 20. Thus the sum is 20.

Problem:
[Q]: What is the sum of all odd numbers in the list 3 83 11 8 5?
[A]:
"""

kwargs = {"max_length": 3200}
math_problem_explanation = get_completion_falcon(fewshot_COT_prompt)
print(math_problem_explanation)

invoking llm at Huggingface Hub
#### User: 
You are a mathematical genius.
The template of the question is given as Q: [question]. The template of the answer is given as A: [answer].
Help me solve this problem with the examples given.

Examples:
[Q]: What is the sum of all odd numbers in the list 0,1,2,3,4,5,6,7,8,9?
[A]: The given numbers are (0,1,2,3,4,5,6,7,8,9). Next select the odd numbers (1,3,5,7,9). Adding all
the selected numbers (1,3,5,7,9) gives 25. Thus the sum is 25. 

[Q]: What is the sum of all odd numbers in the list 22, 35, 100, 3, 75, 2?
[A]: The given numbers are (22, 35, 100, 3, 75, 2). Next select the odd numbers (35,3,75). Adding all
the selected numbers (35,3,75) gives 113. Thus the sum is 113. 

[Q]: What is the sum of all odd numbers in the list 5 2 34 77 14 39?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (5,77,39). Adding all
the selected numbers (5,77,39) gives 121. Thus the sum is 121.

[Q]: What is the sum of all odd numbers in 

In [26]:
# load remote model
from langchain.llms import HuggingFaceHub
model = "tiiuae/falcon-7b-instruct"
# model = "tiiuae/falcon-40b-instruct"
falcon = HuggingFaceHub(
    repo_id=model,
    model_kwargs={"temperature": 2.0,
                  "max_length": 128},
)
     

In [27]:
fewshot_COT_prompt="""You are a mathematical genius.
The template of the question is given as Q: [question]. The template of the answer is given as A: [answer].
Help me solve this problem with the examples given.

Examples:
[Q]: What is the sum of all odd numbers in the list 0,1,2,3,4,5,6,7,8,9?
[A]: The given numbers are (0,1,2,3,4,5,6,7,8,9). Next select the odd numbers (1,3,5,7,9). Adding all
the selected numbers (1,3,5,7,9) gives 25. Thus the sum is 25. 

[Q]: What is the sum of all odd numbers in the list 22, 35, 100, 3, 75, 2?
[A]: The given numbers are (22, 35, 100, 3, 75, 2). Next select the odd numbers (35,3,75). Adding all
the selected numbers (35,3,75) gives 113. Thus the sum is 113. 

[Q]: What is the sum of all odd numbers in the list 5 2 34 77 14 39?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (5,77,39). Adding all
the selected numbers (5,77,39) gives 121. Thus the sum is 121.

[Q]: What is the sum of all odd numbers in the list 1 3 9 7 2 10?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (1,3,9,7). Adding all
the selected numbers (1,3,9,7) gives 20. Thus the sum is 20.

Problem:
[Q]: What is the sum of all odd numbers in the list 3 83 11 8 5?
[A]:
"""

kwargs = {"max_length": 3200}
math_problem_explanation = get_completion_falcon(fewshot_COT_prompt)
print(math_problem_explanation)

invoking llm at Huggingface Hub
#### User: 
You are a mathematical genius.
The template of the question is given as Q: [question]. The template of the answer is given as A: [answer].
Help me solve this problem with the examples given.

Examples:
[Q]: What is the sum of all odd numbers in the list 0,1,2,3,4,5,6,7,8,9?
[A]: The given numbers are (0,1,2,3,4,5,6,7,8,9). Next select the odd numbers (1,3,5,7,9). Adding all
the selected numbers (1,3,5,7,9) gives 25. Thus the sum is 25. 

[Q]: What is the sum of all odd numbers in the list 22, 35, 100, 3, 75, 2?
[A]: The given numbers are (22, 35, 100, 3, 75, 2). Next select the odd numbers (35,3,75). Adding all
the selected numbers (35,3,75) gives 113. Thus the sum is 113. 

[Q]: What is the sum of all odd numbers in the list 5 2 34 77 14 39?
[A]: The given numbers are (5 2 34 77 14 39). Next select the odd numbers (5,77,39). Adding all
the selected numbers (5,77,39) gives 121. Thus the sum is 121.

[Q]: What is the sum of all odd numbers in 

### Discussion 

From all the prompts, I expected the few-shot chain-of-thought prompt to perform the best out of all prompts. The prompt that got a lot closer to the logic and answer was the few-shot prompt, given how it was constructed. The COT and zero-shot prompts provided either with procedures that had little legibility or just a plain, incorrect answer. Overall, the model failed to recognize and extract all the odd numbers from the given list in the question. In all instances, the model did not provide the right sum. 

In this regard, I decreased the temperature of the falcon model using the few shot COT prompt to see if there would be any improvement. I saw little change. Increasing the temperature however, truly pushed the model to provide a non-sensical response. What we can summarize from these experiments is that this falcon model might not be a good model to perform mathematical operations even when few shot COT prompts are used. 